In [11]:
import cv2
import numpy as np
import os

input_folder = "enhanced_frames"
output_folder = "output_pothole"

os.makedirs(output_folder, exist_ok=True)

def detect_pothole(image, output):

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    blur = cv2.GaussianBlur(gray, (7,7), 0)

    mask = cv2.adaptiveThreshold(
        blur, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        21, 5
    )

    kernel_line = np.ones((1, 9), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_line)

    kernel = np.ones((5,5), np.uint8)
    clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)

    kernel_merge = np.ones((15,15), np.uint8)
    merged = cv2.morphologyEx(clean, cv2.MORPH_CLOSE, kernel_merge, iterations=2)

    contours, _ = cv2.findContours(
        merged,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    count = 0

    for cnt in contours:

        area = cv2.contourArea(cnt)

        if area < 1500:
            continue

        x, y, w, h = cv2.boundingRect(cnt)

        aspect_ratio = w / (h + 1e-5)

        if aspect_ratio > 8 or aspect_ratio < 0.15:
            continue

        roi = gray[y:y+h, x:x+w]
        mean_val = np.mean(roi)

        # only reject pure white paint
        if mean_val > 230:
            continue


        if h < 10 or w < 10:
            continue

        cv2.rectangle(output, (x,y), (x+w,y+h), (0,0,255), 3)

        cv2.putText(
            output,
            "Pothole",
            (x, y-5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0,0,255),
            2
        )

        count += 1

    return count

files = sorted(os.listdir(input_folder))

for file in files:

    if not file.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue

    path = os.path.join(input_folder, file)

    image = cv2.imread(path)

    if image is None:
        continue

    image = cv2.resize(image, (640, 480))

    output = image.copy()

    count = detect_pothole(image, output)

    cv2.putText(
        output,
        f"Potholes: {count}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 255),
        2
    )

    cv2.imwrite(
        os.path.join(output_folder, file),
        output
    )

    print(f"{file} → {count} potholes")

print("\nPOTHOLE DETECTION DONE!")

AlligatorCracking_frame_0.jpg → 4 potholes
AlligatorCracking_frame_20.jpg → 4 potholes
AlligatorCracking_frame_35.jpg → 5 potholes
AlligatorCracking_frame_4.jpg → 2 potholes
AlligatorCracking_frame_51.jpg → 3 potholes
AlligatorCracking_frame_9.jpg → 5 potholes
EdgeCracking3_frame_72.jpg → 3 potholes
EdgeCracking4_frame_123.jpg → 0 potholes
EdgeCracking4_frame_129.jpg → 0 potholes
EdgeCracking4_frame_130.jpg → 0 potholes
EdgeCracking4_frame_133.jpg → 0 potholes
EdgeCracking4_frame_144.jpg → 0 potholes
EdgeCracking4_frame_154.jpg → 0 potholes
EdgeCracking9_frame_185.jpg → 0 potholes
EdgeCracking9_frame_202.jpg → 0 potholes
EdgeCracking9_frame_206.jpg → 0 potholes
EdgeCracking9_frame_217.jpg → 0 potholes
RoadPotholes2_frame_522.jpg → 6 potholes
RoadPotholes3_frame_575.jpg → 1 potholes
RoadPotholes3_frame_578.jpg → 1 potholes
RoadPotholes3_frame_585.jpg → 2 potholes
RoadPotholes_frame_478.jpg → 1 potholes
RoadPotholes_frame_479.jpg → 3 potholes
RoadPotholes_frame_491.jpg → 5 potholes
RoadP